In [0]:
checks_failed = []

try:
    top_customers = spark.table("default.top_customers")
    top_products = spark.table("default.top_products")
    monthly_revenue = spark.table("default.monthly_revenue")
except Exception as e:
    raise Exception(f"Quality Check Failed: Could not read Gold tables. Error: {str(e)}")


#Row count checks
customer_rows = top_customers.count()
product_rows = top_products.count()
revenue_rows = monthly_revenue.count()

if customer_rows == 0:
    checks_failed.append("top_customers table is empty")

if product_rows == 0:
    checks_failed.append("top_products table is empty")

if revenue_rows == 0:
    checks_failed.append("monthly_revenue table is empty")


#Null checks
if top_customers.filter("CustomerID IS NULL").count() > 0:
    checks_failed.append("Null CustomerID found in top_customers")

if top_products.filter("Description IS NULL").count() > 0:
    checks_failed.append("Null Description found in top_products")

if monthly_revenue.filter("Month IS NULL").count() > 0:
    checks_failed.append("Null InvoiceMonth found in monthly_revenue")


#Negative value checks
if top_customers.filter("TotalRevenue < 0").count() > 0:
    checks_failed.append("Negative TotalRevenue found in top_customers")

if top_products.filter("TotalRevenue < 0").count() > 0:
    checks_failed.append("Negative TotalRevenue found in top_products")

if monthly_revenue.filter("MonthlyRevenue < 0").count() > 0:
    checks_failed.append("Negative MonthlyRevenue found in monthly_revenue")


# -----------------------------
# 4. Final result
# -----------------------------
if checks_failed:
    raise Exception("Quality Check Failed: " + " | ".join(checks_failed))
else:
    print("All quality checks passed successfully.")
    print(f"top_customers rows: {customer_rows}")
    print(f"top_products rows: {product_rows}")
    print(f"monthly_revenue rows: {revenue_rows}")